# PowderBench house model — trained snowfall post-processing (Colab)

Trains a **LightGBM quantile model** that corrects and blends weather-model
snowfall forecasts at PowderBench stations, using the real lead times you face
in a live round:

* **Truth**: PowderBench's QC'd daily snowfall (`nakas/powderbench-truth`, stations league / SNOTEL).
* **Features**: what GFS, ICON, GEM and NOAA NBM forecast 1, 2 and 3 days ahead
  (Open-Meteo *previous-runs* archive, 2024 →), plus GFS precipitation and temperature,
  station elevation vs. model terrain, location and season.
* **Split by season**: train winter 2024-25, test winter 2025-26 — never random.
* **Scored like the benchmark**: Powder Score vs. the official climatology, next to
  each raw model and the model median (`baseline-consensus`).

Runtime: CPU is fine. Open-Meteo's free tier allows ~10k "calls"/day per IP; the
feature download for ~150 stations x 2 winters fits in one day and is cached to
Google Drive so re-runs are free.

In [ ]:
!pip -q install lightgbm scikit-learn pandas pyarrow requests huggingface_hub joblib
import os, json, time, math, hashlib
from datetime import date, timedelta
import numpy as np, pandas as pd, requests
from huggingface_hub import hf_hub_download, list_repo_files

try:  # cache API responses on Drive so a re-run (or tomorrow's run) is free
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/powderbench_cache'
except Exception:
    CACHE = './powderbench_cache'
os.makedirs(CACHE, exist_ok=True)
N_NETWORK = 100          # network stations sampled on top of the 45 featured
SEASONS = {'train': ('2024-11-01', '2025-04-15'), 'test': ('2025-11-01', '2026-04-15')}
MODELS = ['gfs_seamless', 'icon_seamless', 'gem_seamless', 'ncep_nbm_conus']

## 1 · Truth and stations

In [ ]:
REPO = 'nakas/powderbench-truth'
files = [f for f in list_repo_files(REPO, repo_type='dataset') if f.startswith('stations/truth_2024') or f.startswith('stations/truth_2025') or f.startswith('stations/truth_2026')]
truth = pd.concat([pd.read_parquet(hf_hub_download(REPO, f, repo_type='dataset')) for f in files])
truth['date'] = pd.to_datetime(truth['date']).dt.date
meta = pd.read_parquet(hf_hub_download(REPO, 'stations/stations.parquet', repo_type='dataset'))
conus = meta[(meta.latitude.between(24, 50)) & (meta.longitude.between(-125, -66))]
featured = conus[conus.tier.fillna('featured') == 'featured']
network = conus[conus.tier == 'network'].sample(min(N_NETWORK, (conus.tier == 'network').sum()), random_state=0)
sts = pd.concat([featured, network]).drop_duplicates('station_id').reset_index(drop=True)
print(len(sts), 'stations;', truth.valid.mean().round(3), 'valid share')

# the official climatology (what Powder Score is measured against)
climo = pd.read_csv('https://raw.githubusercontent.com/andrewnakas/powderbench/main/data/climatology/stations.csv.gz')

## 2 · Forecasts at real lead times (Open-Meteo previous-runs)

In [ ]:
def om(params):
    key = hashlib.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()[:24]
    path = f'{CACHE}/{key}.json'
    if os.path.exists(path):
        return json.load(open(path))
    for attempt in range(6):
        r = requests.get('https://previous-runs-api.open-meteo.com/v1/forecast', params=params, timeout=180)
        if r.status_code == 429:
            print('rate-limited — sleeping 65s'); time.sleep(65); continue
        r.raise_for_status(); break
    d = r.json(); json.dump(d, open(path, 'w')); time.sleep(2.5)
    return d

HOURLY = ([f'snowfall_previous_day{l}' for l in (1, 2, 3)]
          + [f'precipitation_previous_day{l}' for l in (1, 2, 3)]
          + [f'temperature_2m_previous_day{l}' for l in (1, 2, 3)])

def station_features(row, start, end):
    p = {'latitude': row.latitude, 'longitude': row.longitude, 'elevation': row.elevation_ft * 0.3048,
         'hourly': ','.join(HOURLY), 'models': ','.join(MODELS), 'timezone': 'auto',
         'start_date': start, 'end_date': end}
    d = om(p)
    h = pd.DataFrame(d['hourly']); h['date'] = pd.to_datetime(h['time']).dt.date
    agg = {c: ('mean' if c.startswith('temperature') else 'sum') for c in h.columns if c not in ('time', 'date')}
    day = h.groupby('date').agg(agg)
    # keep every model's snowfall, plus GFS precip/temperature (phase errors)
    keep = [c for c in day.columns if c.startswith('snowfall') or c.endswith('gfs_seamless')]
    # snowfall cm -> in, precipitation mm -> in, temperature stays degC
    div = [2.54 if k.startswith('snowfall') else 25.4 if k.startswith('precipitation') else 1.0 for k in keep]
    day = day[keep] / np.array(div)
    day['grid_elev_diff_m'] = row.elevation_ft * 0.3048 - d.get('elevation', np.nan)
    day['station_id'] = row.station_id
    return day.reset_index()

frames = []
for name, (a, b) in SEASONS.items():
    for i, row in enumerate(sts.itertuples()):
        try:
            frames.append(station_features(row, a, b).assign(season=name))
        except Exception as e:
            print('skip', row.station_id, e)
        if i % 25 == 0: print(name, i, '/', len(sts))
fc = pd.concat(frames, ignore_index=True)
fc.to_parquet(f'{CACHE}/features.parquet')
fc.shape

## 3 · Training rows for the three horizons

A round starting on day D predicts day D with the forecast issued the day
before (lead 1), D+1 with lead 2, D+2 with lead 3. The 48h/72h targets are
cumulative, exactly as scored.

In [ ]:
fc['date'] = pd.to_datetime(fc['date']).dt.date
t = truth[truth.station_id.isin(sts.station_id)].set_index(['station_id', 'date'])
F = fc.set_index(['station_id', 'date'])
geo = sts.set_index('station_id')[['latitude', 'longitude', 'elevation_ft']]
c50 = climo.set_index(['station_id', 'doy'])

def snow_cols(lead):  # each model's snowfall at this lead
    return [f'snowfall_previous_day{lead}_{m}' for m in MODELS]

rows = []
for (sid, d), f0 in F.iterrows():
    feats = {'station_id': sid, 'date': d, 'season': f0['season'], 'grid_elev_diff_m': f0['grid_elev_diff_m'],
             'doy': d.timetuple().tm_yday, **geo.loc[sid].to_dict()}
    ok, truth_cum, model_cum = True, 0.0, np.zeros(len(MODELS))
    for k, h in enumerate((24, 48, 72)):
        dk = d + timedelta(days=k)
        if (sid, dk) not in F.index or (sid, dk) not in t.index or not t.loc[(sid, dk), 'valid']:
            break
        fk = F.loc[(sid, dk)]
        model_cum = model_cum + fk[snow_cols(k + 1)].to_numpy(dtype=float)
        truth_cum += t.loc[(sid, dk), 'snow24_in']
        r = dict(feats, horizon_h=h, truth=truth_cum)
        r.update({f'cum_{m}': v for m, v in zip(MODELS, model_cum)})
        r.update({f'precip_lead{k+1}': fk.get(f'precipitation_previous_day{k+1}_gfs_seamless', np.nan),
                  f'temp_lead{k+1}': fk.get(f'temperature_2m_previous_day{k+1}_gfs_seamless', np.nan)})
        r['model_median'] = float(np.nanmedian(model_cum)); r['model_spread'] = float(np.nanstd(model_cum))
        doy = min(d.timetuple().tm_yday, 365)
        r['climo_p50'] = c50.loc[(sid, doy), f'h{h}_p50'] if (sid, doy) in c50.index else np.nan
        rows.append(r)
D = pd.DataFrame(rows)
FEATS = [c for c in D.columns if c.startswith(('cum_', 'precip_', 'temp_')) or c in
         ('model_median', 'model_spread', 'grid_elev_diff_m', 'doy', 'latitude', 'longitude', 'elevation_ft', 'climo_p50')]
print(D.shape, D.groupby(['season', 'horizon_h']).size())

## 4 · Train quantile models (one per horizon x quantile)

In [ ]:
import lightgbm as lgb
QS = {'p10': .1, 'p25': .25, 'p50': .5, 'p75': .75, 'p90': .9}
models = {}
for h in (24, 48, 72):
    tr = D[(D.season == 'train') & (D.horizon_h == h)]
    for name, q in QS.items():
        m = lgb.LGBMRegressor(objective='quantile', alpha=q, n_estimators=400, learning_rate=0.03,
                              num_leaves=31, min_child_samples=40, subsample=0.8, subsample_freq=1,
                              colsample_bytree=0.8, verbose=-1)
        m.fit(tr[FEATS], tr['truth'])
        models[(h, name)] = m
print('trained', len(models), 'models')

## 5 · Test season: Powder Score next to the raw models

In [ ]:
te = D[D.season == 'test'].copy()
for h in (24, 48, 72):
    sel = te.horizon_h == h
    for name in QS:
        te.loc[sel, name] = np.clip(models[(h, name)].predict(te.loc[sel, FEATS]), 0, None)
qcols = list(QS)
te[qcols] = np.sort(te[qcols].to_numpy(), axis=1)  # monotone quantiles

def powder(pred, rows):
    ok = rows['climo_p50'].notna() & pred.notna()
    mae, ref = (pred[ok] - rows.truth[ok]).abs().mean(), (rows.climo_p50[ok] - rows.truth[ok]).abs().mean()
    return round(100 * (1 - mae / ref), 1), round(mae, 3)

board = {'house (LightGBM p50)': powder(te['p50'], te), 'model median (consensus)': powder(te['model_median'], te)}
for m in MODELS:
    board[m] = powder(te[f'cum_{m}'], te)
board['climatology'] = powder(te['climo_p50'], te)
pd.DataFrame(board, index=['Powder Score', 'MAE in']).T.sort_values('Powder Score', ascending=False)

In [ ]:
# probabilistic track (pinball over the 5 quantiles) and Powder Alert (P >= 6" in 24h)
def pinball(rows):
    l = [np.where(rows.truth >= rows[c], q * (rows.truth - rows[c]), (q - 1) * (rows.truth - rows[c])) for c, q in QS.items()]
    return float(np.mean(l))
print('pinball (test):', round(pinball(te), 4))
for h in (24, 48, 72):
    print(h, 'h', powder(te.loc[te.horizon_h == h, 'p50'], te[te.horizon_h == h]),
          'vs consensus', powder(te.loc[te.horizon_h == h, 'model_median'], te[te.horizon_h == h]))

## 6 · Save the model

The bundle (models + feature list) goes to your Drive and, if you log in, to a
Hugging Face model repo so the daily submitter can load it.

In [ ]:
import joblib
bundle = {'models': models, 'features': FEATS, 'models_used': MODELS, 'trained_on': SEASONS['train'],
          'test_scores': board}
path = f'{CACHE}/powderbench_house_gbm.joblib'
joblib.dump(bundle, path)
print('saved', path)
# optional: share it (needs a write token)
# from huggingface_hub import login, HfApi; login()
# HfApi().create_repo('nakas/powderbench-house-gbm', exist_ok=True)
# HfApi().upload_file(path_or_fileobj=path, path_in_repo='powderbench_house_gbm.joblib', repo_id='nakas/powderbench-house-gbm')